# Flight Delay Data Management and Machine Learning

This notebook works with the FAA flight dataset from 2013 to:
1. Cleanse the data by identifying and replacing null values with zero
2. Create a machine learning model to predict flight delays (>15 minutes) based on day of week and airport
3. Save the trained model to a file for external use
4. Create a CSV file containing all airports and their associated IDs

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pickle

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("Libraries imported successfully!")

In [ ]:
# Load the flight data
df_flights = pd.read_csv('data/flights.csv')
print(f"Dataset shape: {df_flights.shape}")
print(f"\nColumn names:\n{df_flights.columns.tolist()}")

In [ ]:
# Display first few rows to understand the data structure
print("First 5 rows of the dataset:")
df_flights.head()

In [ ]:
# Get basic information about the dataset
print("Dataset info:")
df_flights.info()
print("\nBasic statistics:")
df_flights.describe()

In [ ]:
# Check for null values in all columns
print("Null values per column:")
null_counts = df_flights.isnull().sum()
print(null_counts)

# Show only columns with null values
columns_with_nulls = null_counts[null_counts > 0]
if len(columns_with_nulls) > 0:
    print(f"\nColumns with null values:\n{columns_with_nulls}")
else:
    print("\nNo null values found in the dataset.")

In [ ]:
# If there are null values, investigate and replace with appropriate values
# Based on the requirement, we'll replace null values with zero

if df_flights.isnull().sum().sum() > 0:
    print("Before cleaning:")
    print(df_flights.isnull().sum())
    
    # Replace all null values with 0
    df_flights.fillna(0, inplace=True)
    
    print("\nAfter cleaning (replacing nulls with 0):")
    print(df_flights.isnull().sum())
    print("\nData cleaning completed!")
else:
    print("No null values to clean.")

In [ ]:
# Explore the target variable (ArrDel15 - arrival delay more than 15 minutes)
print("Distribution of arrival delays (ArrDel15):")
print(df_flights['ArrDel15'].value_counts())
print(f"\nPercentage of delayed flights: {(df_flights['ArrDel15'].mean() * 100):.2f}%")

# Visualize the distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df_flights, x='ArrDel15')
plt.title('Distribution of Flight Delays (ArrDel15)')
plt.xlabel('Delayed (0=No, 1=Yes)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Explore delays by day of week
delay_by_day = df_flights.groupby('DayOfWeek')['ArrDel15'].agg(['count', 'sum', 'mean']).reset_index()
delay_by_day.columns = ['DayOfWeek', 'TotalFlights', 'DelayedFlights', 'DelayRate']

# Add day names for better understanding
day_names = {1: 'Monday', 2: 'Tuesday', 3: 'Wednesday', 4: 'Thursday', 5: 'Friday', 6: 'Saturday', 7: 'Sunday'}
delay_by_day['DayName'] = delay_by_day['DayOfWeek'].map(day_names)

print("Delay rates by day of week:")
print(delay_by_day)

# Visualize delay rates by day of week
plt.figure(figsize=(10, 6))
sns.barplot(data=delay_by_day, x='DayName', y='DelayRate')
plt.title('Flight Delay Rate by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Delay Rate (Proportion)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Explore delays by destination airport (top 10 airports with most flights)
airport_stats = df_flights.groupby(['DestAirportID', 'DestAirportName'])['ArrDel15'].agg(['count', 'sum', 'mean']).reset_index()
airport_stats.columns = ['DestAirportID', 'DestAirportName', 'TotalFlights', 'DelayedFlights', 'DelayRate']
airport_stats = airport_stats.sort_values('TotalFlights', ascending=False)

print("Top 10 airports by total flights and their delay rates:")
print(airport_stats.head(10))

In [ ]:
# Prepare data for machine learning model
# Features: DayOfWeek and DestAirportID
# Target: ArrDel15 (arrival delay more than 15 minutes)

X = df_flights[['DayOfWeek', 'DestAirportID']]
y = df_flights['ArrDel15']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")
print(f"Target variable: ArrDel15")

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"\nTraining set delay rate: {y_train.mean():.4f}")
print(f"Testing set delay rate: {y_test.mean():.4f}")

In [ ]:
# Create and train the logistic regression model
model = LogisticRegression(solver='lbfgs', random_state=42, max_iter=1000)

print("Training the logistic regression model...")
model.fit(X_train, y_train)
print("Model training completed!")

# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print(f"\nModel coefficients: {model.coef_[0]}")
print(f"Model intercept: {model.intercept_[0]}")

In [ ]:
# Evaluate the model performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Test the model with a specific example
# Example: Predict delay probability for Monday (1) to Las Vegas (12892)
# Note: We'll find an actual airport ID from the data

# Get a sample airport ID for testing
sample_airport_id = df_flights['DestAirportID'].iloc[0]
sample_airport_name = df_flights['DestAirportName'].iloc[0]

# Test prediction for Monday (DayOfWeek=1) to the sample airport
test_input = [[1, sample_airport_id]]
delay_probability = model.predict_proba(test_input)[0]

print(f"Sample prediction:")
print(f"Flight on Monday (1) to {sample_airport_name} (ID: {sample_airport_id})")
print(f"Probability of NO delay: {delay_probability[0]:.4f} ({delay_probability[0]*100:.2f}%)")
print(f"Probability of DELAY: {delay_probability[1]:.4f} ({delay_probability[1]*100:.2f}%)")

In [ ]:
# Save the trained model to a pickle file for external use
model_filename = 'flight_delay_model.pkl'

print(f"Saving the trained model to {model_filename}...")
with open(model_filename, 'wb') as file:
    pickle.dump(model, file)

print(f"Model saved successfully to {model_filename}")

# Test loading the model to ensure it works
print("\nTesting model loading...")
with open(model_filename, 'rb') as file:
    loaded_model = pickle.load(file)

# Verify the loaded model works
test_prediction = loaded_model.predict_proba(test_input)[0]
print(f"Loaded model test prediction: {test_prediction}")
print("Model loading test successful!")

In [ ]:
# Create a CSV file with unique airports and their IDs
# Using both origin and destination airports to get complete list

# Get unique origin airports
origin_airports = df_flights[['OriginAirportID', 'OriginAirportName']].drop_duplicates()
origin_airports.columns = ['AirportID', 'AirportName']

# Get unique destination airports
dest_airports = df_flights[['DestAirportID', 'DestAirportName']].drop_duplicates()
dest_airports.columns = ['AirportID', 'AirportName']

# Combine and remove duplicates
all_airports = pd.concat([origin_airports, dest_airports]).drop_duplicates()
all_airports = all_airports.sort_values('AirportName').reset_index(drop=True)

print(f"Total unique airports found: {len(all_airports)}")
print("\nFirst 10 airports:")
print(all_airports.head(10))

# Save to CSV file
airports_filename = 'airports.csv'
all_airports.to_csv(airports_filename, index=False)
print(f"\nAirports data saved to {airports_filename}")

In [ ]:
# Summary of completed tasks
print("=" * 60)
print("SUMMARY OF COMPLETED TASKS")
print("=" * 60)
print("? 1. Data cleansing: Identified and replaced null values with zero")
print("? 2. Model creation: Built logistic regression model to predict flight delays")
print(f"   - Model accuracy: {accuracy:.4f}")
print(f"   - Features: Day of Week and Destination Airport ID")
print(f"   - Target: Arrival delay more than 15 minutes (ArrDel15)")
print(f"? 3. Model export: Saved trained model to '{model_filename}'")
print(f"? 4. Airport data: Created '{airports_filename}' with {len(all_airports)} unique airports")
print("\nAll requirements have been successfully completed!")